# 7. 现代卷积神经网络

在本章中的每一个模型都曾一度占据主导地位，其中许多模型都是ImageNet竞赛的优胜者。

> ImageNet竞赛自2010年以来，一直是计算机视觉中监督学习进展的指向标。

这些模型包括：

*   **AlexNet**。它是第一个在大规模视觉竞赛中击败传统计算机视觉模型的大型神经网络；

*   **使用重复块的网络（VGG）**。它利用许多重复的神经网络块；

*   **网络中的网络（NiN）**。它重复使用由卷积层和 $1 \times 1$ 卷积层（用来代替全连接层）来构建深层网络;

*   **含并行连结的网络（GoogLeNet）**。它使用并行连结的网络，通过不同窗口大小的卷积层和最大汇聚层来并行抽取信息；

*   **残差网络（ResNet）**。它通过残差块构建跨层的数据通道，是计算机视觉中最流行的体系架构；

*   **稠密连接网络（DenseNet）**。它的计算成本很高，但给我们带来了更好的效果。

虽然深度神经网络的概念非常简单——将神经网络堆叠在一起。但由于不同的网络架构和超参数选择，这些神经网络的性能会发生很大变化。本章介绍的神经网络是将人类直觉和相关数学见解结合后，经过大量研究试错后的结晶。 

---


## 7.1 深度卷积神经网络 (AlexNet)

> 如果说 LeNet 是神经网络的“童年”，那么 **AlexNet** 就是神经网络的“成年礼”。2012 年，AlexNet 在 ImageNet 竞赛中以绝对优势夺冠，彻底终结了传统计算机视觉的统治，开启了深度学习的黄金时代。

### 1. 核心动机：为什么 LeNet 不够了？

LeNet 在处理手写数字（28x28）时非常完美，但在面对真实世界的自然图像（如 ImageNet 的高清图）时遇到了瓶颈：

1.  **特征复杂度**：真实图像的纹理、光照和背景极其复杂，LeNet 的参数量不足以捕捉这些特征。
2.  **计算资源**：2012 年左右，GPU 算力开始爆发，使得训练更深、更宽的网络成为可能。
3.  **激活函数限制**：Sigmoid 在深层网络中极易导致**梯度消失**。

---

### 2. AlexNet 的五大核心创新

1.  **更深的网络结构**：8 层加权层（5 层卷积 + 3 层全连接）。
2.  **使用 ReLU 激活函数**：用 $\text{max}(0, x)$ 代替 Sigmoid。ReLU 在正区间导数为 1，彻底解决了深层网络的梯度消失问题，且计算速度极快。
3.  **使用 Dropout (暂退法)**：在最后的全连接层引入 Dropout（我们在 4.6 节学过），有效控制了巨大参数量带来的过拟合。
4.  **数据增广 (Data Augmentation)**：通过随机裁剪、翻转和亮度调整，人为增加训练样本，提高模型鲁棒性。
5.  **最大汇聚 (Max Pooling)**：用最大汇聚代替 LeNet 的平均汇聚，更能突出图像的显著特征。

---

### 3. 构建 AlexNet

AlexNet 的输入通常是 $224 \times 224$。由于 Fashion-MNIST 只有 $28 \times 28$，我们在加载数据时需要将其**强制放大**。


In [6]:
import torch
from torch import nn

def get_alexnet_model(num_classes: int = 10) -> nn.Sequential:
    """构建 AlexNet 模型。通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。
    
    结构参考 2012 年经典论文，针对 Fashion-MNIST 输出 10 类。
    """
    net = nn.Sequential(
        # 第一块：大尺寸卷积核提取粗略特征
        # 输入：(Batch, 1, 224, 224)
        nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2),  # -> (Batch, 64, 55, 55)
        nn.ReLU(),
        # 重叠汇聚：
        #   - 缓解过拟合;
        #   - 增大感受野，并避免信息通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。丢失过快
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 64, 27, 27)

        # 第二块：减小核尺寸，增加通道数
        # 输入：(Batch, 64, 27, 27)
        nn.Conv2d(64, 192, kernel_size=5, padding=2),           # -> (Batch, 192, 27, 27)
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 192, 13, 13)

        # 第三块：连续三个卷积层提取精细特征
        # 输入：(Batch, 192, 13, 13)
        nn.Conv2d(192, 384, kernel_size=3, padding=1),          # -> (Batch, 384, 13, 13)
        nn.ReLU(),
        nn.Conv2d(384, 256, kernel_size=3, padding=1),          # -> (Batch, 256, 13, 13)
        nn.ReLU(),
        nn.Conv2d(256, 256, kernel_size=3, padding=1),          # -> (Batch, 256, 13, 13)
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2),                  # -> (Batch, 256, 6, 6)

        # 第四块：全连接层 + Dropout
        nn.Flatten(),                                           # -> (Batch, 256 * 6 * 6)
        # 参数量主要集中在该全连接层
        nn.Linear(256 * 6 * 6, 4096),                           # -> (Batch, 4096)
        nn.ReLU(),
        # Dropout 抑制过拟合
        nn.Dropout(p=0.5),
        # 4096 -> 4096 是当时认为的力大砖飞，
        # 现在更倾向于通过更深的卷积层和更轻的输出层来达到同样甚至更好的效果。
        nn.Linear(4096, 4096),                                  # -> (Batch, 4096)
        nn.ReLU(),
        nn.Dropout(p=0.5),

        # 输出层
        nn.Linear(4096, num_classes)                            # -> (Batch, num_classes)
    )
    return net

# --- 验证维度流转 ---
X = torch.randn(1, 1, 224, 224)
net = get_alexnet_model()
for layer in net:
    X = layer(X)
    print(f"{layer.__class__.__name__:15} 输出形状: {X.shape}")

Conv2d          输出形状: torch.Size([1, 64, 55, 55])
ReLU            输出形状: torch.Size([1, 64, 55, 55])
MaxPool2d       输出形状: torch.Size([1, 64, 27, 27])
Conv2d          输出形状: torch.Size([1, 192, 27, 27])
ReLU            输出形状: torch.Size([1, 192, 27, 27])
MaxPool2d       输出形状: torch.Size([1, 192, 13, 13])
Conv2d          输出形状: torch.Size([1, 384, 13, 13])
ReLU            输出形状: torch.Size([1, 384, 13, 13])
Conv2d          输出形状: torch.Size([1, 256, 13, 13])
ReLU            输出形状: torch.Size([1, 256, 13, 13])
Conv2d          输出形状: torch.Size([1, 256, 13, 13])
ReLU            输出形状: torch.Size([1, 256, 13, 13])
MaxPool2d       输出形状: torch.Size([1, 256, 6, 6])
Flatten         输出形状: torch.Size([1, 9216])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([

---

### 4. 训练 AlexNet

> 由于训练 ImageNet 模型花费时间较长，因而，仅出于演示目的，我们使用 `Fashion-MNIST` 进行对 AlexNet 的训练。

为了适配 AlexNet，我们需要修改之前的 `load_fashion_mnist` 函数，加入 `Resize` 步骤。


In [7]:
import d2l_utils as d2l
from torch.utils import data

def load_data_alexnet(batch_size: int, resize: int = 224) -> tuple[data.dataloader, data.dataloader]:
    """加载 Fashion-MNIST 并调整尺寸以适配 AlexNet。"""
    return d2l.load_fashion_mnist(batch_size=batch_size, resize=resize)

# 建议配置
batch_size = 128
train_iter, test_iter = load_data_alexnet(batch_size)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data


In [ ]:
import d2l_utils as d2l
def train_alexnet( # TODO: 带封装到 d2l_utils
    net: nn.Module,
    train_iter: data.DataLoader,
    test_iter: data.DataLoader,
    num_epochs: int,
    lr: float,
    device: torch.device
) -> None:
    """训练函数，用以演示 AlexNet 模型的训练过程。"""

    # 1. 初始化权重、搬运到 GPU
    def init_weights(m: nn.Module):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            # 对输出层使用 Xavier 初始化（如果不接 ReLU）
            # 注意：这里的逻辑依赖于 net 的结构定义，如果最后一层是 net[-1]
            if m == net[-1]:
                nn.init.xavier_uniform_(m.weight)
            else:
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    net.apply(init_weights)
    net.to(device)

    # 2. 优化器与损失函数
    # 按照论文配置 momentum 和 weight_decay
    optimizer = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9, weight_decay=0.0005)
    loss_fn = nn.CrossEntropyLoss()

    print(f"在设备 {device} 上启动训练...")

    # 3. 训练循环
    for epoch in range(num_epochs):
        net.train()
        metric = d2l.Accumulator(3, device) # [train_loss_sum, train_acc_sum, n]

        for X, y in train_iter:
            optimizer.zero_grad()
            X, y = X.to(device), y.to(device)
            
            y_hat = net(X)
            l = loss_fn(y_hat, y)
            l.backward()
            optimizer.step()

            with torch.no_grad():
                metric.add(l * X.shape[0], d2l.count_correct_tensor(y_hat, y), X.shape[0])

        # 评估测试集
        train_loss = (metric[0]/metric[2]).item()
        train_acc = (metric[1]/metric[2]).item()
        test_acc = d2l.evaluate_accuracy_gpu(net, test_iter, device)
        print(f"Epoch {epoch+1} | Loss: {train_loss:.3f} | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f}")

# 运行示例
batch_size = 128 # 如果你的显卡显存小于 8GB，建议将 batch_size 设为 64 或 32
lr, num_epochs = 0.01, 10 # 建议先用较小的学习率观察收敛情况
device = d2l.get_default_device()

train_iter, test_iter = load_data_alexnet(batch_size)

net = get_alexnet_model()

train_alexnet(net, train_iter, test_iter, num_epochs, lr, device)

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
在设备 cuda 上启动训练...
Epoch 1 | Loss: 0.609 | Train Acc: 0.780 | Test Acc: 0.866
Epoch 2 | Loss: 0.333 | Train Acc: 0.879 | Test Acc: 0.893
Epoch 3 | Loss: 0.280 | Train Acc: 0.898 | Test Acc: 0.904
Epoch 4 | Loss: 0.249 | Train Acc: 0.909 | Test Acc: 0.909
Epoch 5 | Loss: 0.228 | Train Acc: 0.917 | Test Acc: 0.905
Epoch 6 | Loss: 0.206 | Train Acc: 0.924 | Test Acc: 0.918
Epoch 7 | Loss: 0.193 | Train Acc: 0.929 | Test Acc: 0.919
Epoch 8 | Loss: 0.179 | Train Acc: 0.935 | Test Acc: 0.918
Epoch 9 | Loss: 0.167 | Train Acc: 0.938 | Test Acc: 0.924
Epoch 10 | Loss: 0.157 | Train Acc: 0.942 | Test Acc: 0.924


---

### 5. 细致梳理：AlexNet 的数学与逻辑

1.  **计算量与参数量**：
    *   AlexNet 虽然只有 8 层，但参数量高达 **6000 万**个。
    *   绝大部分参数集中在第一个全连接层：$256 \times 6 \times 6 \times 4096 \approx 3700$ 万。
2.  **ReLU 的胜利**：
    *   ReLU 的计算就是一行 `if x > 0`，而 Sigmoid 涉及复杂的指数运算。这使得 AlexNet 的训练速度比同规模的 Sigmoid 网络快 6 倍以上。
3.  **为什么第一层用 11x11 的大核？**
    *   在输入图片很大（224x224）时，需要一个足够大的窗口来捕捉最初的物体轮廓。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

*   **显存预警**：AlexNet 的全连接层非常吃显存。如果你的显卡显存较小（如 4GB），请务必调小 `batch_size`（如 64 或 32）。
*   **学习率调优**：由于 AlexNet 比较深，建议初始学习率设小一点（如 0.01），并配合 Adam 优化器。

---

### 7. 总结 Checklist

*   **ReLU**：明白它为什么能取代 Sigmoid（解决梯度消失）。
*   **Dropout**：明白它在 AlexNet 这种大参数模型中的必要性。
*   **Resize**：掌握如何处理输入尺寸不匹配的问题。
*   **架构感**：记住 AlexNet “卷积-卷积-卷积卷积卷积-全连接”的节奏。

---


## 7.2 使用块的网络（VGG）

> 如果说 AlexNet 证明了“深”是有用的，那么 VGG 则确立了深度神经网络设计的**工业标准**：**模块化**。
> 
> VGG（由牛津大学视觉几何组 "Visual Geometry Group" 提出）不再纠结于每一层该用多大的核，而是发明了“VGG 块”的概念，通过重复堆叠相同的块来构建极深的网络。

### 1. 核心设计哲学：模块化

在 VGG 之前，网络设计像是在“做手工”，每一层的核大小、步幅都可能不同（如 AlexNet）。

**VGG 的创新**：
1.  **统一算子**：全线使用 $3 \times 3$ 卷积核和 $2 \times 2$ 最大汇聚层。
2.  **堆叠逻辑**：多个 $3 \times 3$ 卷积连续堆叠，其感受野等价于更大的卷积核（如 2 个 $3 \times 3$ 等价于 1 个 $5 \times 5$），但参数更少，非线性激活更多。
3.  **通道倍增**：每经过一个汇聚层，图像宽高减半，但输出通道数翻倍。

---

### 2. 构建 VGG块

In [9]:
import torch
from torch import nn, Tensor

def vgg_block(num_convs: int, in_channels: int, out_channels: int) -> nn.Sequential:
    """构建一个 VGG 块。

    每个 VGG 块包含若干个填充为 1 的 3x3 卷积层（ReLU 激活），
    最后接一个步幅为 2 的 2x2 最大汇聚层。

    Args:
        num_convs: 块内卷积层的数量。
        in_channels: 输入通道数。
        out_channels: 输出通道数。

    Returns:
        nn.Sequential: 组装好的 VGG 块。
        
        一个 VGG 块执行完后，输出形状变为：(Batch, C_{out}, H // 2, W // 2)
    """
    layers: list[nn.Module] = []
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
        in_channels = out_channels # 后续卷积层的输入等于前一层的输出
    
    layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
    return nn.Sequential(*layers)

---

### 3. 构建 VGG-11 网络 

> 11 是指 8 卷积层 + 3 全连接层


In [10]:
def get_vgg_model(conv_arch: list[tuple[int, int]], in_channels: int = 1, num_classes: int = 10) -> nn.Sequential:
    """根据指定的架构列表构建 VGG 网络。

    Args:
        conv_arch: 
            - 架构列表，如 [(1, 64), (1, 128), (2, 256), (2, 512), (2, 512)]。
            - 元组内容：(卷积层数, 输出通道数)
        in_channels: 图像输入通道数。
        num_classes: 分类类别数。

    以 Fashion-MNIST 分类为例。

    Returns:
        nn.Sequential: 完整的 VGG 网络。
    """
    vgg_layers: list[nn.Module] = []
    
    # 1. 卷积块部分
    # 输入：(Batch, in_channels, 224, 224) 
    #   224 是因为 AlexNet 通常使用，感觉是一种习惯
    for (num_convs, out_channels) in conv_arch:
        vgg_layers.append(vgg_block(num_convs, in_channels, out_channels))
        in_channels = out_channels
        
    # 2. 全连接层部分
    # 设 经过 n 个 VGG 块
    # 则，输入为 (Batch, last_out_channels, 224 // (2 ** n), 224 // (2 ** n))
    side = 7 # 此处仅用以演示 224 // (2 ** 5)，写的是 VGG-11 的定义
    net = nn.Sequential(
        *vgg_layers,
        nn.Flatten(),   # -> (Batch, out_channels * side * side)
        nn.Linear(out_channels * side * side, 4096), nn.ReLU(), nn.Dropout(0.5), # -> (Batch, 4096)
        nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(0.5), # -> (Batch, 4096)
        nn.Linear(4096, num_classes) # (Batch, num_classes)
    )
    return net

# 定义标准的 VGG-11 架构
vgg11_arch = [(1, 64), (1, 128), (2, 256), (2, 512), (2, 512)]
net = get_vgg_model(vgg11_arch)

---

### 4. 训练 VGG-11


In [11]:
# 瘦身：因为参数量较大

# 将所有通道数除以 4，大幅减少计算量
ratio = 4
small_vgg_arch = [(pair[0], pair[1] // ratio) for pair in vgg11_arch]
net = get_vgg_model(small_vgg_arch)

# --- 验证维度 ---
X = torch.randn(size=(1, 1, 224, 224))
for blk in net:
    X = blk(X)
    print(f"{blk.__class__.__name__:15} 输出形状: {X.shape}")

Sequential      输出形状: torch.Size([1, 16, 112, 112])
Sequential      输出形状: torch.Size([1, 32, 56, 56])
Sequential      输出形状: torch.Size([1, 64, 28, 28])
Sequential      输出形状: torch.Size([1, 128, 14, 14])
Sequential      输出形状: torch.Size([1, 128, 7, 7])
Flatten         输出形状: torch.Size([1, 6272])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([1, 4096])
ReLU            输出形状: torch.Size([1, 4096])
Dropout         输出形状: torch.Size([1, 4096])
Linear          输出形状: torch.Size([1, 10])


In [12]:
# 该条不建议运行，感觉会比较慢
# 主要是学习新的 CNN 思想

import d2l_utils as d2l

# 超参数建议
lr, num_epochs, batch_size = 0.05, 10, 128
train_iter, test_iter = d2l.load_fashion_mnist(batch_size, resize=224)

# 执行训练，借用上面的函数
train_alexnet(net, train_iter, test_iter, num_epochs, lr, d2l.get_default_device())

Dataset check complete at: /home/august/deepseek/pytorch_study/temp/data
在设备 cuda 上启动训练...
Epoch 1 | Loss: 0.913 | Train Acc: 0.662 | Test Acc: 0.856
Epoch 2 | Loss: 0.350 | Train Acc: 0.871 | Test Acc: 0.883
Epoch 3 | Loss: 0.293 | Train Acc: 0.892 | Test Acc: 0.902
Epoch 4 | Loss: 0.259 | Train Acc: 0.904 | Test Acc: 0.910
Epoch 5 | Loss: 0.236 | Train Acc: 0.913 | Test Acc: 0.909
Epoch 6 | Loss: 0.216 | Train Acc: 0.920 | Test Acc: 0.914
Epoch 7 | Loss: 0.203 | Train Acc: 0.926 | Test Acc: 0.920
Epoch 8 | Loss: 0.192 | Train Acc: 0.929 | Test Acc: 0.920
Epoch 9 | Loss: 0.183 | Train Acc: 0.933 | Test Acc: 0.919
Epoch 10 | Loss: 0.172 | Train Acc: 0.936 | Test Acc: 0.923


### 5. 细致梳理：VGG 的数学与逻辑

1.  **为什么用 $3 \times 3$ 核？**
    *   两个 $3 \times 3$ 卷积层的参数量是 $2 \times (3 \times 3) = 18$。
    *   一个 $5 \times 5$ 卷积层的参数量是 $1 \times (5 \times 5) = 25$。
    *   **结论**：用多个小核代替大核，可以在保持感受野一致的情况下，**减少参数量**并**增加非线性层数**（每一层都有一个 ReLU）。
2.  **深度与复杂度的权衡**：
    *   VGG 证明了通过单纯地增加深度（VGG-16, VGG-19），模型的识别能力会稳步提升。
3.  **内存瓶颈**：
    *   VGG 的前两个全连接层占用了绝大部分参数量（约 1 亿个），这是现代架构（如 ResNet）致力于优化的方向。

---

### 6. 补充知识

`nn.Sequential` vs `nn.ModuleList` 的核心区别

| 特性 | `nn.Sequential` | `nn.ModuleList` |
| :--- | :--- | :--- |
| **执行逻辑** | **自动化**：内部自动实现了按顺序执行的 `forward`。 | **手动化**：只负责注册模块，`forward` 必须你自己写循环。 |
| **灵活性** | **低**：只能按严格的线性顺序执行。 | **高**：可以在循环中加入 `if/else`、跳跃连接（Skip Connection）等逻辑。 |
| **代码简洁度** | 极简，适合简单的堆叠。 | 略显繁琐，需要定义类。 |
| **用途场景** | 经典的 VGG 块、MLP 层。 | **更复杂的架构**（如 ResNet 的层管理、含有变长子模块的网络）。 |

为什么要有这两种组件？

1.  **参数注册 (Parameter Registration)**：
    两者最重要的共同点是都能**自动将列表内的参数注册到主模型中**。如果你只用 Python 原生的 `list`（例如 `self.layers = [nn.Linear(10, 10)]`），当你调用 `model.to(device)` 时，这些层**不会**被移动到 GPU。
    
2.  **VGG 的选择**：
    对于 VGG 这种“一条路走到黑”的线性架构，`nn.Sequential` 是最佳选择，因为它减少了样板代码。
    
3.  **ModuleList 的优势**：
    如果你在设计一个不确定深度的网络，或者在 `forward` 过程中需要用到中间层的输出（比如目标检测中的特征金字塔），`nn.ModuleList` 通过下标索引（`self.layers[i]`）的方式会非常方便。

---

### 7. 总结 Checklist

*   **VGG 块**：理解其内部构造（多个卷积 + 一个汇聚）。
*   **模块化设计**：明白如何通过改变 `arch` 列表来快速生成不同深度的网络。
*   **感受野等效性**：记住 $3 \times 3$ 堆叠的优势。
*   **显存意识**：知道为什么在实验中需要使用 `ratio` 进行缩放。

---


## 7.3 网络中的网络 (NiN)

> 如果说 VGG 解决了“如何模块化堆叠层”的问题，那么 NiN 则提出了一个更深刻的问题：卷积层之后一定要接笨重的全连接层吗？
>
> NiN 引入了两个革命性的概念：
> 
> - $1 \times 1$卷积层（用于增加非线性）和全局平均汇聚层（用于彻底取代全连接层）。

### 1. 设计动机

1.  **全连接层的弊端**：在 AlexNet 和 VGG 中，最后的全连接层占用了模型 90% 以上的参数量。这不仅容易导致过拟合，还极大地增加了显存负担。
2.  **局部结构的复杂性**：传统的卷积层只是做线性变换。NiN 认为，应该在每个像素位置应用一个“微型网络”来提取更复杂的特征。

---

### 2. 核心组件：NiN 块 (NiN Block)

一个 NiN 块由一个普通卷积层后接两个 $1 \times 1$ 卷积层组成。
*   **$1 \times 1$ 卷积的作用**：它充当了跨通道的全连接层，能够学习通道之间的复杂交互，同时完全保留空间结构。

---

### 3. 构建 NiN 块


In [15]:
import torch
from torch import nn, Tensor

def nin_block(
    in_channels: int, 
    out_channels: int, 
    kernel_size: int, 
    strides: int, 
    padding: int
) -> nn.Sequential:
    """构建一个 NiN 块。

    包含一个标准卷积层和两个 1x1 卷积层（充当全连接层）。
    
    Args:
        in_channels: 输入通道数。
        out_channels: 输出通道数。
        kernel_size: 第一层卷积的核大小。
        strides: 第一层卷积的步幅。
        padding: 第一层卷积的填充。

    Returns:
        nn.Sequential: 组装好的 NiN 块。
    """

    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, strides, padding),
        nn.ReLU(),
        # 第一个 1x1 卷积：增加非线性，不改变空间维度
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        # 第二个 1x1 卷积：进一步提取通道间特征
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

---

### 4. NiN 网络架构

> NiN 彻底取消了最后那两个 4096 维的全连接层，改为使用**全局平均汇聚 (Global Average Pooling, GAP)**。


In [18]:
def get_nin_net(num_classes: int = 10) -> nn.Sequential:
    """构建 NiN 网络模型。"""
    net = nn.Sequential(
        # 块 1: 模拟 AlexNet 的第一层
        nin_block(1, 96, kernel_size=11, strides=4, padding=0),
        nn.MaxPool2d(3, stride=2),
        
        # 块 2: 增加深度
        nin_block(96, 256, kernel_size=5, strides=1, padding=2),
        nn.MaxPool2d(3, stride=2),
        
        # 块 3: 进一步提取特征
        nin_block(256, 384, kernel_size=3, strides=1, padding=1),
        nn.MaxPool2d(3, stride=2),
        nn.Dropout(0.5),
        
        # 块 4: 最后一块的输出通道数等于类别数
        nin_block(384, num_classes, kernel_size=3, strides=1, padding=1),
        
        # 全局平均汇聚层 (GAP)
        # 将 (Batch, 10, 5, 5) 变为 (Batch, 10, 1, 1)
        nn.AdaptiveAvgPool2d((1, 1)),
        
        # 展平后直接输出，不需要任何 Linear 层！
        nn.Flatten()
    )
    return net

In [19]:
# --- 验证维度流转 ---
X = torch.randn(size=(1, 1, 224, 224))
net = get_nin_net()
for layer in net:
    X = layer(X)
    print(f"{layer.__class__.__name__:20} 输出形状: {X.shape}")

Sequential           输出形状: torch.Size([1, 96, 54, 54])
MaxPool2d            输出形状: torch.Size([1, 96, 26, 26])
Sequential           输出形状: torch.Size([1, 256, 26, 26])
MaxPool2d            输出形状: torch.Size([1, 256, 12, 12])
Sequential           输出形状: torch.Size([1, 384, 12, 12])
MaxPool2d            输出形状: torch.Size([1, 384, 5, 5])
Dropout              输出形状: torch.Size([1, 384, 5, 5])
Sequential           输出形状: torch.Size([1, 10, 5, 5])
AdaptiveAvgPool2d    输出形状: torch.Size([1, 10, 1, 1])
Flatten              输出形状: torch.Size([1, 10])



---

### 5. 细致梳理：NiN 的数学与逻辑

1.  **$1 \times 1$ 卷积的本质**：
    *   它在空间维度上是“点乘”，在通道维度上是“全连接”。
    *   **优点**：引入了更多的非线性激活函数，增强了网络的表达能力，且参数量极低。
2.  **全局平均汇聚 (GAP) 的优势**：
    *   **参数压缩**：直接将最后一个特征图的每个通道求平均值作为分类得分。
    *   **正则化**：GAP 没有任何参数，天然防止了过拟合。
    *   **空间鲁棒性**：对输入图像的空间平移更加鲁棒。
3.  **架构对比**：
    *   **AlexNet/VGG**：卷积层提取特征 $\to$ 全连接层做分类。
    *   **NiN**：NiN 块提取高度抽象特征 $\to$ GAP 直接输出类别得分。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

*   **`nn.AdaptiveAvgPool2d((1, 1))`**：这是实现 GAP 的标准 PyTorch 方式。无论输入特征图是 $5 \times 5$ 还是 $7 \times 7$，它都能自动计算步幅和核大小，将其压缩为 $1 \times 1$。
*   **训练建议**：由于 NiN 没有庞大的全连接层，它的训练速度通常比 AlexNet 快，且对显存的要求更低。建议学习率设为 0.1。

---

### 7. 补充知识

#### 7.1 核心定义

**消融实验 (Ablation Study)** 是指**从一个完整的、效果很好的系统中，移除某个特定的组件/特性（如某种层、某种损失函数、某个超参数），观察系统的表现会下降多少。**

如果移除它后效果大幅变差，说明这个组件是核心贡献者；如果效果没变，说明这个组件可能只是“凑数”的。

#### 7.2 消融实验 vs. 普通对比实验

*   **对比实验 (Comparative Study)**：通常是“横向”的。比如 A 算法和 B 算法哪个好？它是为了证明 **“我比别人强”**。
*   **消融实验 (Ablation Study)**：通常是“纵向”的。比如我的算法里有 3 个创新点，如果去掉创新点 1 会怎样？它是为了证明 **“我的每一个部分都有用”**。


#### 7.3 以 NiN 为例

> **实验设计**：将 NiN 最后的全局平均汇聚 (GAP) 换回 VGG 那样的两个 4096 维全连接层。

*   **系统 A (原版 NiN)**：使用 NiN 块 + GAP。
*   **系统 B (消融版)**：使用 NiN 块 + **全连接层**（相当于“消融”掉了 GAP 这个特性）。

**通过对比 A 和 B：**
1.  如果 B 的准确率大幅下降或产生严重过拟合，就证明了 **GAP（全局平均汇聚）在防止过拟合和特征抽象方面的卓越贡献**。
2.  这就是在验证 GAP 这个特定组件的价值。

#### 7.4 为什么要写消融实验？

在发表 AI 论文或进行复杂项目汇报时，消融实验是必不可少的，因为它能回答两个硬核问题：
1.  **拒绝“玄学”**：证明你的模型变好不是运气，而是因为这些设计的确科学。
2.  **定位核心**：弄清楚到底哪一步改进最关键，从而在资源有限时知道该保留什么。

**小结**：消融实验就是 **“拆零件，看机器还能不能跑、跑得顺不顺”** 的对比过程。

---

### 8. 总结 Checklist

*   **NiN 块**：理解 1x1 卷积如何充当“像素级全连接”。
*   **GAP**：理解为什么它可以取代全连接层。
*   **参数效率**：明白为什么 NiN 的参数量远小于 VGG。
*   **设计范式**：学会这种“纯卷积”架构的设计思路。

---


## 7.4 含并行连结的网络 (GoogLeNet)

> 如果说 NiN 告诉我们卷积层可以变得多复杂，那么 GoogLeNet 则在 2014 年凭一己之力将神经网络推向了“多路径并行”的新高度。
>
> GoogLeNet 的核心创新是引入了 **Inception 块**。
>
> 它的设计哲学是：**与其纠结于该用 $1\times1$、$3\times3$ 还是 $5\times5$ 的卷积核，不如全都要，让网络自己通过训练决定哪些路径更重要。**

---

### 1. 核心动机：多尺度特征与计算效率

在设计 CNN 时，确定合适的卷积核大小（Kernel Size）通常需要大量实验。

GoogLeNet 的核心贡献者提出了 **Inception 块**，解决了两个核心问题：

1. 尺度多样性：同一张图片中，感兴趣的物体尺寸不一（有的占满全图，有的只是角落的一点）。并行路径允许模型在同一层同时捕捉高频细节（小核）和低频轮廓（大核）。

2. 参数爆炸：直接堆叠大卷积核会消耗海量算力。GoogLeNet 引入 $1 \times 1$ 卷积作为“瓶颈层（Bottleneck）”来压缩通道数。

---

### 2. Inception 块的深度拆解

一个标准的 Inception 块包含四条并行路径，最终在通道维度上进行合并。

#### 2.1 结构逻辑

- 路径 1：纯 $1 \times 1$ 卷积。提取跨通道相关性。
- 路径 2：$1 \times 1$ 卷积 (压缩) -> $3 \times 3$ 卷积。
- 路径 3：$1 \times 1$ 卷积 (压缩) -> $5 \times 5$ 卷积。
- 路径 4：$3 \times 3$ 最大汇聚 -> $1 \times 1$ 卷积 (调整通道数)。

这是 GoogLeNet 最精妙的原子组件。注意：为了让输出在空间维度上能拼接，所有分支都使用了适当的 `padding`。

#### 2.2 构建 Inception 块

In [2]:
import torch
from torch import nn, Tensor
from typing import Union

class Inception(nn.Module):
    """GoogLeNet 的核心组件：Inception 块。
    
    该模块通过四个并行路径提取多尺度特征，
    并利用 1x1 卷积降低计算复杂度。
    """
    def __init__(
        self, 
        in_channels: int, 
        c1: int, 
        c2: tuple[int, int], 
        c3: tuple[int, int], 
        c4: int
    ) -> None:
        """
        Args:
            in_channels: 输入通道数。
            c1: 路径 1 (1x1 conv) 的输出通道数。
            c2: 路径 2 (1x1 conv 降维, 3x3 conv) 的输出通道数。
            c3: 路径 3 (1x1 conv 降维, 5x5 conv) 的输出通道数。
            c4: 路径 4 (3x3 max pool, 1x1 conv 调整通道数) 的输出通道数。
        """
        super().__init__()

        # 路径 1：单 1x1 卷积
        self.p1_1 = nn.Conv2d(in_channels, c1, kernel_size=1)
        
        # 路径 2：1x1 卷积降维 -> 3x3 卷积提取特征 (padding=1 保证尺寸不变)
        self.p2_1 = nn.Conv2d(in_channels, c2[0], kernel_size=1)
        self.p2_2 = nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1)
        
        # 路径 3：1x1 卷积降维 -> 5x5 卷积提取特征 (padding=2 保证尺寸不变)
        self.p3_1 = nn.Conv2d(in_channels, c3[0], kernel_size=1)
        self.p3_2 = nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2)
        
        # 路径 4：3x3 最大汇聚 (stride=1, padding=1 保证尺寸不变) -> 1x1 卷积降维
        self.p4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.p4_2 = nn.Conv2d(in_channels, c4, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        """执行并行计算并在通道维度 (dim=1) 拼接。"""
        branch1 = torch.relu(self.p1_1(x))
        branch2 = torch.relu(self.p2_2(torch.relu(self.p2_1(x))))
        branch3 = torch.relu(self.p3_2(torch.relu(self.p3_1(x))))
        branch4 = torch.relu(self.p4_2(self.p4_1(x)))
        
        # 将四个分支的结果在通道维度 (dim=1) 上合并 
        # 输出为：(Batch, C1+C2+C3+C4, H, W)
        return torch.cat((branch1, branch2, branch3, branch4), dim=1)


---

### 3. GoogLeNet 架构：分段组装

GoogLeNet 巧妙地通过五个模块化的阶段（Stages）来构建，逐渐增加通道数并减小空间分辨率。

#### 3.1 阶段详细定义

1. Stage 1 & 2 (Stem)：类似于 AlexNet 的开头，使用大卷积核快速降低分辨率。
2. Stage 3：引入 Inception 块，开始提取多尺度特征。
3. Stage 4：网络最深的部分，连续堆叠 5 个 Inception 块。
4. Stage 5：最后两个 Inception 块，接全局平均汇聚层（GAP）。

#### 3.2 构建 GoogLeNet


In [3]:
def get_googlenet(num_classes: int = 10) -> nn.Sequential:
    """构建标准 GoogLeNet 架构。"""
    
    # 阶段 1 & 2: 初始特征提取
    b1 = nn.Sequential(
        nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3), nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    )
    b2 = nn.Sequential(
        nn.Conv2d(64, 64, kernel_size=1), nn.ReLU(),
        nn.Conv2d(64, 192, kernel_size=3, padding=1), nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    )
    
    # 阶段 3: 两个 Inception 块
    b3 = nn.Sequential(
        Inception(192, 64, (96, 128), (16, 32), 32),   # 输出 256
        Inception(256, 128, (128, 192), (32, 96), 64), # 输出 480
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    )
    
    # 阶段 4: 五个 Inception 块
    b4 = nn.Sequential(
        Inception(480, 192, (96, 208), (16, 48), 64),
        Inception(512, 160, (112, 224), (24, 64), 64),
        Inception(512, 128, (128, 256), (24, 64), 64),
        Inception(512, 112, (144, 288), (32, 64), 64),
        Inception(528, 256, (160, 320), (32, 128), 128),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    )
    
    # 阶段 5: 最终特征合并与输出
    b5 = nn.Sequential(
        Inception(832, 256, (160, 320), (32, 128), 128),
        Inception(832, 384, (192, 384), (48, 128), 128),
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten()
    )
    
    return nn.Sequential(b1, b2, b3, b4, b5, nn.Linear(1024, num_classes))

In [4]:
# --- 验证 ---
X = torch.randn(size=(1, 1, 96, 96)) # 为了演示，使用 96x96 输入
net = get_googlenet()
for blk in net:
    X = blk(X)
    print(f"{blk.__class__.__name__:15} 输出形状: {X.shape}")

Sequential      输出形状: torch.Size([1, 64, 24, 24])
Sequential      输出形状: torch.Size([1, 192, 12, 12])
Sequential      输出形状: torch.Size([1, 480, 6, 6])
Sequential      输出形状: torch.Size([1, 832, 3, 3])
Sequential      输出形状: torch.Size([1, 1024])
Linear          输出形状: torch.Size([1, 10])



---

### 4. 细致梳理：GoogLeNet 的数学与逻辑

1.  **为什么需要四个分支？**
    *   图像中的特征大小不一。有的特征很小（需 $3\times3$），有的很大（需 $5\times5$）。并行分支让网络可以在同一层提取**不同尺度**的特征。
2.  **$1\times1$ 卷积的“瓶颈”作用**：
    *   **不加 $1\times1$**：$5\times5$ 卷积在 192 个通道上计算量极大。
    *   **加上 $1\times1$**：先将 192 通道压缩到 16 通道，再做 $5\times5$ 卷积。参数量和计算量通常能减少到原来的 **1/10**。
3.  **全局平均汇聚 (GAP)**：
    *   继承了 NiN 的思想，彻底抛弃了 AlexNet 那种庞大的全连接层，使 GoogLeNet 虽然层数极深，但总参数量却比 AlexNet 小得多。
    *   同时 GAP 天然具有抗过拟合作用。

---

### 5. 关键工程暗知识 (Engineering Wisdom)

*   **辅助分类器 (Auxiliary Classifiers)**：
    *   在原始论文中，由于网络太深，中间层很难得到梯度 (所以上面的 GoogLeNet 可能会欠拟合)。GoogLeNet 在 Stage 4 增加了两个辅助分类头，用于在训练期间向中间层注入梯度，防止梯度消失。
    *   **备注**：随着 Batch Norm（下一节）的普及，现代版本的 GoogLeNet 已经不再需要辅助分类器了。
*   **计算量监控**：
    *   Inception 块虽然强大，但分支过多会增加显卡调度开销。在实际工程中，有时简单的 VGG 堆叠在某些硬件上反而更快。

---

### 6. VGG vs GoogLeNet

VGG 和 GoogLeNet 是同一时代的“双雄”（都在 2014 年的 ImageNet 大赛中大放异彩），而它们的设计思想代表了深度学习演化过程中的**两个截然不同的极端方向**。

我们可以把它们比作 **“暴力美学的重装坦克”** 与 **“精密计算的轻量化战机”**。

#### 6.1 核心矛盾：深度 vs. 效率

虽然两者的目标都是“让网络更深”，但策略完全不同：

1. VGG 的思想：简单、纯粹、暴力堆叠
   *   **设计逻辑**：既然深有用，那我就用最简单的积木（$3 \times 3$ 卷积）一直叠下去。
   *   **特点**：结构极度规整，全是串行连接。
   *   **代价**：**参数量巨大**。VGG-16 有 1.38 亿个参数，显存占用极高。
   *   **工程隐喻**：这是一种“力大砖飞”的思维，靠堆硬件和深度来换精度。

2. GoogLeNet 的思想：复杂、并行、极致省钱
   *   **设计逻辑**：深有用，但不能这么浪费。我得在每一层都用最少的参数榨取最多的特征。
   *   **特点**：结构极其复杂（Inception 块），大量使用并行分支和 $1 \times 1$ 卷积。
   *   **成就**：**参数量极小**。GoogLeNet 只有 680 万个参数，**仅为 VGG 的 1/20**，但精度反而更高。
   *   **工程隐喻**：这是一种“以巧破千斤”的思维，通过复杂的架构设计来弥补算力的不足。

#### 6.2 深度对比：设计维度的差异

| 维度 | VGG (2014) | GoogLeNet (2014) |
| :--- | :--- | :--- |
| **连接方式** | **串行 (Serial)**：一根筋到底。 | **并行 (Parallel)**：四条路径同时看。 |
| **卷积核选择** | **单一**：全线 $3 \times 3$。 | **多样**：$1\times1, 3\times3, 5\times5$ 全都要。 |
| **全连接层** | **重度依赖**：最后有 3 个巨大的 FC 层。 | **彻底抛弃**：使用全局平均汇聚 (GAP)。 |
| **参数量** | 约 138,000,000 (极重) | 约 6,800,000 (极轻) |
| **核心贡献** | 证明了**模块化设计**的威力。 | 证明了**模型架构搜索 (NAS)** 的潜力。 |

#### 6.3 演化意义

在深度学习历史上，这两个模型分别开启了后世的两大主流流派：

1.  **VGG 开启了“结构化堆叠”流派**：
    后来的 ResNet（7.6节）继承了 VGG 的简洁美学，使用统一的算子，通过“残差连接”解决了 VGG 太深了练不动的痛点。
2.  **GoogLeNet 开启了“多分支/轻量化”流派**：
    后来的 MobileNet、ShuffleNet 都是 Inception 思想的后代，致力于在手机、嵌入式芯片上跑深度学习，追求极致的推理速度和低功耗。

#### 6.4 工程视角：两者的代码风格对比

如果你在写一个通用的训练框架，你会发现：

*   **VGG 型代码**：非常适合用 `nn.Sequential` 编写，逻辑清晰，是入门者的教科书。
*   **GoogLeNet 型代码**：必须继承 `nn.Module` 自定义 `forward` 函数，因为 `nn.Sequential` 处理不了那种“分叉再合并”的 `torch.cat` 逻辑。

```python
# VGG 风格：线性美感
vgg_style = nn.Sequential(nn.Conv2d(...), nn.Conv2d(...), nn.MaxPool2d(...))

# GoogLeNet 风格：图结构美感
class InceptionStyle(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        return torch.cat([b1, b2], dim=1) # 这种分叉是 VGG 不具备的
```

#### 6.5 小结

*   **VGG** 告诉我们：**“深”是王道**，积木要统一。
*   **GoogLeNet** 告诉我们：**“巧”是王道**，参数要省着用。

---

### 7. 总结 Checklist

*   **Inception 块**：掌握其四个分支的构成和 $1\times1$ 卷积的降维作用。
*   **维度对齐**：理解 `padding` 如何保证不同分支输出的宽高一致。
*   **拼接操作**：理解 `torch.cat(..., dim=1)` 在通道维度的合并。
*   **多尺度特征**：明白其通过多路径获取多样化感受野的原理。

---


## 7.5 批量规范化 (Batch Normalization)

> 如果说前几节是在研究如何改变“积木的形状”（网络架构），那么这一节则是研究如何改良“积木的材质”（数值稳定性）。
> 
> Batch Normalization 是深度学习历史上最伟大的发明之一，它解决了深层网络极难训练的痛点，允许我们使用更大的学习率和更随意的初始化。

### 1. 核心动机：为什么需要 BN？

1.  **内部协变量偏移 (Internal Covariate Shift)**：随着网络加深，底层参数的微小变化会导致高层输入分布的剧烈波动，迫使高层不断适应新的分布，导致训练极慢。
2.  **梯度消失/爆炸**：BN 将激活值拉回到均值 0、方差 1 附近，避开了激活函数（如 Sigmoid 或 Tanh）的饱和区。
3.  **正则化效果**：BN 在计算时引入了小批量（Mini-batch）的噪声，起到了一定的正则化作用，有时可以省去 Dropout。

---

### 2. 数学原理：BN 的四步曲

对于一个 Batch 中的数据 $B = \{x_1, \dots, x_m\}$，BN 执行以下转换：

1.  **计算均值**：$\mu_B = \frac{1}{m} \sum_{i=1}^m x_i$
2.  **计算方差**：$\sigma_B^2 = \frac{1}{m} \sum_{i=1}^m (x_i - \mu_B)^2$
3.  **规范化**：$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$ （$\epsilon$ 是为了防止除以 0 的小常数）
4.  **缩放与平移 (关键！)**：$y_i = \gamma \hat{x}_i + \beta$
    *   $\gamma$（缩放）和 $\beta$（平移）是**可学习的参数**，他们的形状与 $x$ 相同。它们允许模型在必要时恢复原始分布的表达能力。

---

### 3. 从零实现 BN 层

实现 BN 最难的地方在于区分**训练模式**（算 Batch 统计量）和**预测模式**（用全局移动平均量）。

- 在训练过程中，我们无法得知使用整个数据集来估计平均值和方差，所以只能根据每个小批次的平均值和方差不断训练模型。
- 而在预测模式下，可以根据整个数据集精确计算批量规范化所需的平均值和方差。


In [10]:
import torch
from torch import nn, Tensor

def batch_norm(
    X: Tensor,
    gamma: Tensor, 
    beta: Tensor, 
    moving_mean: Tensor, 
    moving_var: Tensor, 
    eps: float, 
    momentum: float,
    is_training: bool
) -> tuple[Tensor, Tensor, Tensor]:
    """批量规范化 (Batch Normalization) 的底层算子实现。
    
    该函数实现了对输入张量的规范化操作，根据当前是否处于训练模式，
    自动切换使用小批量统计量或全局移动平均统计量。

    Args:
        X: 输入张量。如果是全连接层，形状为 (batch_size, num_features)；
            如果是卷积层，形状为 (batch_size, num_channels, height, width)。
        gamma: 训练过程中可学习的拉伸 (scale) 参数。
        beta: 训练过程中可学习的偏移 (shift) 参数。
        moving_mean: 整个训练过程中维护的全局移动平均均值。
        moving_var: 整个训练过程中维护的全局移动平均方差。
        eps: 为了保持数值稳定性（防止除以 0）而添加到分母中的微小常数。
        momentum: 更新移动平均统计量时使用的动量因子（通常取 0.9）。
        is_training: 是否处于训练模式。

    Returns:
        一个包含三个元素的元组：
        - Y: 规范化、缩放和平移后的输出张量，形状与 X 相同。
        - moving_mean: 更新后的全局移动平均均值。
        - moving_var: 更新后的全局移动平均方差。

    Raises:
        AssertionError: 当输入张量 X 的维度既不是 2 也不是 4 时抛出。
    """
    # 通过 is_grad_enabled 判断当前是训练模式还是预测模式
    if not is_training:
        # 预测模式：直接使用传入的移动平均所得的均值和方差
        X_hat = (X - moving_mean) / torch.sqrt(moving_var + eps)
    else:
        # 训练模式
        assert len(X.shape) in (2, 4)
        if len(X.shape) == 2:
            # 全连接层：在特征维度(列)计算均值和方差
            mean = X.mean(dim=0)
            var = ((X - mean) ** 2).mean(dim=0)
        else:
            # 卷积层：在通道维度(dim=1)计算均值和方差
            # 保持形状 (1, C, 1, 1) 以便广播
            mean = X.mean(dim=(0, 2, 3), keepdim=True)
            var = ((X - mean) ** 2).mean(dim=(0, 2, 3), keepdim=True)

        # 训练时使用当前 Batch 的统计量进行规范化
        X_hat = (X - mean) / torch.sqrt(var + eps)
        
        # 更新移动平均值（用于未来的预测模式）
        # 使用 [:] 原地修改，保持引用一致
        with torch.no_grad():
            moving_mean[:] = (1.0 - momentum) * moving_mean + momentum * mean
            moving_var[:] = (1.0 - momentum) * moving_var + momentum * var

    # 缩放和平移
    Y = gamma * X_hat + beta
    return Y, moving_mean.detach(), moving_var.detach()

In [11]:
class BatchNorm(nn.Module):
    """自定义批量规范化 (Batch Normalization) 层。

    该层实现了对输入数据的规范化，即通过减去均值并除以标准差来调整数据分布，
    并引入了可学习的拉伸参数 (gamma) 和偏移参数 (beta)。

    Attributes:
        gamma: 形状为 (1, num_features) 或 (1, num_features, 1, 1) 的可学习参数。
        beta: 形状与 gamma 相同的可学习参数。
        moving_mean: 整个训练期间维护的全局移动平均均值，不参与梯度竞争。
        moving_var: 整个训练期间维护的全局移动平均方差，不参与梯度竞争。
    """
    def __init__(self, num_features: int, num_dims: int):
        """初始化 BatchNorm 层。

        Args:
            num_features: 特征数量。对于全连接层是输出维数，对于卷积层是输出通道数。
            num_dims: 输入张量的总维度。
                - 2 表示全连接层，输入形状为 (batch_size, num_features)；
                - 4 表示卷积层，输入形状为 (batch_size, num_features, height, width)。

        Raises:
            ValueError: 当 num_dims 不在 [2, 4] 集合中时。
        """
        super().__init__()
        shape = (1, num_features) if num_dims == 2 else (1, num_features, 1, 1)
        # 参与梯度竞争的参数
        self.gamma = nn.Parameter(torch.ones(shape))
        self.beta = nn.Parameter(torch.zeros(shape))
        # 不参与梯度竞争的统计变量（只追踪数值，不存入计算图）
        self.register_buffer('moving_mean', torch.zeros(shape))
        self.register_buffer('moving_var', torch.ones(shape))

    def forward(self, X: Tensor) -> Tensor:
        # 保证设备一样
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(X.device)
            self.moving_var = self.moving_var.to(X.device)
            
        # 显式更新 moving_mean， moving_var
        Y, self.moving_mean, self.moving_var = batch_norm(
            X, self.gamma, self.beta, self.moving_mean,
            self.moving_var, eps=1e-5, momentum=0.1,
            is_training=self.training)
        return Y

In [13]:
# --- 验证代码 ---

# 1. 初始化 (模拟 3 通道卷积)
bn = BatchNorm(3, 4).cuda()
X = torch.ones(2, 3, 2, 2).cuda() * 10  # 输入全为 10

# 2. 训练模式：验证全局均值是否更新
bn.train()
_ = bn(X)
print(f"训练更新后的全局均值 (应为 1.0): {bn.moving_mean.unique().item():.1f}") 
# 计算: 0(初始) * 0.9 + 10(当前) * 0.1 = 1.0

# 3. 预测模式：验证是否停止更新并使用全局量
bn.eval()
_ = bn(X * 2) # 输入翻倍，但预测模式不应受此影响
print(f"预测模式下的全局均值 (应保持 1.0): {bn.moving_mean.unique().item():.1f}")

# 4. 设备检查
print(f"显存一致性: {bn.moving_mean.is_cuda}")

训练更新后的全局均值 (应为 1.0): 1.0
预测模式下的全局均值 (应保持 1.0): 1.0
显存一致性: True



---

### 4. 简洁实现：使用 PyTorch 内置层

在实际工程中，我们直接使用 `nn.BatchNorm1d`（用于全连接）或 `nn.BatchNorm2d`（用于卷积）。


In [14]:
def get_bn_lenet() -> nn.Sequential:
    """使用内置 BN 层改造 LeNet。"""
    return nn.Sequential(
        nn.Conv2d(1, 6, kernel_size=5, bias=False), nn.BatchNorm2d(6), nn.Sigmoid(),
        nn.AvgPool2d(kernel_size=2, stride=2),
        nn.Conv2d(6, 16, kernel_size=5, bias=False), nn.BatchNorm2d(16), nn.Sigmoid(),
        nn.AvgPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Linear(16 * 4 * 4, 120, bias=False), nn.BatchNorm1d(120), nn.Sigmoid(),
        nn.Linear(120, 84, bias=False), nn.BatchNorm1d(84), nn.Sigmoid(),
        nn.Linear(84, 10)
    )


---

### 5. 细致梳理：BN 的知识点补充

1.  **BN 放在哪？**
    *   **经典争论**：是放在激活函数前（Conv -> BN -> ReLU）还是激活函数后（Conv -> ReLU -> BN）？
    *   **教材观点**：通常放在**卷积层/全连接层之后，激活函数之前**。
2.  **预测模式的特殊性**：
    *   在测试单张图片时，Batch Size 为 1，无法计算方差。因此，必须使用训练时记录的“移动平均值”。
3.  **为什么 BN 允许更大的学习率？**
    *   它消除了参数缩放对梯度量级的影响。如果不加 BN，权重翻倍会导致梯度翻倍；加了 BN 后，权重的缩放会被规范化步骤抵消，训练更稳。
4.  **为什么 BN 不适合极小的 Batch Size？**
    *   如果 `batch_size=1`，方差为 0，BN 就失效了。通常建议 `batch_size >= 16`。

---

### 6. 关键工程暗知识 (Engineering Wisdom)

*   **Bias 的冗余性**：如果一个卷积层后面紧跟 BN 层，那么卷积层的 `bias` 参数可以设为 `False`。
    *   **原因**：BN 的第三步会减去均值 $\mu_B$，这会把卷积层的偏置 $b$ 抵消掉。最后的偏置由 BN 层的可学习参数 $\beta$ 负责。
    *   **代码**：`nn.Conv2d(..., bias=False)`。
    *   类似的，**Linear** 的 `bias` 参数也可以设为 `False`。 
*   **Momentum 参数**：PyTorch 中 BN 的 `momentum` 默认是 0.1。它控制了旧信息保留的比例。

---

### 7. 总结 Checklist

*   **均值方差计算**：理解全连接（按特征）与卷积（按通道）的区别。
*   **训练 vs 预测**：深刻理解为什么测试时行为不同。
*   **可学习参数**：记住 $\gamma$ 和 $\beta$ 的作用。
*   **加速原理**：明白 BN 如何缓解梯度消失。

---


## 7.6 残差网络 (ResNet)

> 在 2015 年之前，深度学习界被一个诡异的现象困扰：当网络深到一定程度（比如超过 20 层）时，准确率不仅不再提升，反而会急剧下降。
> 
> 这并不是因为过拟合（训练误差也在上升），而是因为深层网络极难优化，梯度在成百上千层中穿梭时逐渐迷失。
>
> **ResNet** 的出现彻底解决了这个问题，它通过一根简单的“跳跃连接”，让我们可以训练成百上千层的神经网络（如 ResNet-152）。

### 1. 核心动机：网络退化与残差学习

#### 1.1 为什么深层网络会“退化”？

在传统网络（如 VGG）中，我们希望每一层学到的映射为 $f(x)$。
*   **痛点**：若该层其实不需要做任何变换（即最优映射是恒等映射 $f(x) = x$），在存在非线性激活和随机权重的约束下，模型很难通过参数的精确拟合来实现这一恒等变换。
*   **退化现象**：这不是由于过拟合（过拟合是训练好、测试差），而是由于深层网络**极难优化**，导致训练集上的准确率都上不去。

#### 1.2 残差学习 (Residual Learning) 的破局

ResNet 的核心思想是：与其让网络学习一个完整的映射 $f(x)$，不如让它学习 **残差（Residual）映射 $g(x) = f(x) - x$**。
*   **逻辑**：如果恒等映射是最优的，网络只需要通过权重衰减将 $g(x)$ 的权重推向 0 即可（因为有 L2 正则化/权重衰减，这容易很多）。
*   **跳跃连接 (Skip Connection)**：将输入 $x$ 直接“跨层”加到输出上。最终输出为 $y = \text{relu}(g(x) + x)$。

#### 1.3 跳跃连接 (Skip Connection)

在反向传播时，对于 $y = g(x) + x$，其导数为 $\frac{\partial y}{\partial x} = \frac{\partial g}{\partial x} + 1$。
*   这个 **“+1”** 保证了即使深层卷积部分的梯度 $\frac{\partial g}{\partial x}$ 变得非常微小，梯度依然能顺着这根“电梯线”传回底层，彻底解决了**梯度消失**问题。

---

### 2. 构建残差块 (Residual Block)

残差块是 ResNet 的最小原子单位。我们需要处理两种情况：
1.  **恒等映射**：输入和输出维度一致。
2.  **投影映射**：输入输出维度不一致（通道翻倍或分辨率减半），此时需用 $1 \times 1$ 卷积调整 $x$ 的形状。


In [21]:
import torch
from torch import nn, Tensor
from typing import Optional

class Residual(nn.Module):
    """残差块的实现。

    该模块实现了 ResNet 的核心逻辑：y = g(x) + x。
    包含两个 3x3 卷积层，并支持通过 1x1 卷积调整输入维度以匹配输出。

    Attributes:
        conv1 (nn.Conv2d): 第一层卷积。
        conv2 (nn.Conv2d): 第二层卷积。
        conv3 (Optional[nn.Conv2d]): 用于调整形状的 1x1 投影卷积。
        bn1 (nn.BatchNorm2d): 第一层批规范化。
        bn2 (nn.BatchNorm2d): 第二层批规范化。
    """

    def __init__(
        self, 
        input_channels: int, 
        num_channels: int, 
        use_1x1conv: bool = False, 
        strides: int = 1
    ) -> None:
        """初始化残差块。

        Args:
            input_channels (int): 输入张量的通道数。
            num_channels (int): 输出张量的通道数。
            use_1x1conv (bool): 是否使用 1x1 卷积来改变通道数或减小分辨率。
            strides (int): 第一层卷积的步幅，用于下采样。
        """
        super().__init__()
        
        # 第一层卷积：可能包含下采样（stride > 1）
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.bn1 = nn.BatchNorm2d(num_channels)
        
        # 第二层卷积：保持形状
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_channels)

        # 投影路径：如果输入输出形状不匹配，则使用 1x1 卷积对齐
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1, stride=strides)
        else:
            self.conv3 = None

        self.relu = nn.ReLU()

    def forward(self, x: Tensor) -> Tensor:
        """残差块的前向传播逻辑。

        Args:
            x (Tensor): 输入张量，形状为 (Batch, C, H, W)。

        Returns:
            Tensor: 相加并激活后的张量。
        """
        # 主路径计算
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        
        # 计算跳跃路径（Shortcut）
        if self.conv3 is not None:
            x = self.conv3(x)
            
        # 逐元素相加 (Element-wise addition)
        return self.relu(y + x)


---

### 3. 构建 ResNet-18 网络架构

ResNet-18 包含 1 个初始层、4 个残差阶段（每个阶段 2 个块）和 1 个输出层。


In [22]:
def resnet_block(
    input_channels: int,
    num_channels: int,
    num_residuals: int,
    first_block: bool = False
) -> nn.Sequential:
    """构建 ResNet 的一个阶段（Stage）。

    Args:
        input_channels (int): 输入通道数。
        num_channels (int): 该阶段输出的通道数。
        num_residuals (int): 该阶段包含的残差块数量。
        first_block (bool): 是否是全网络的第一个阶段（通常不进行下采样）。

    Returns:
        nn.Sequential: 组装好的阶段模块。
    """
    blk = []
    for i in range(num_residuals):
        if i == 0 and not first_block:
            # 阶段的第一个块：进行下采样并增加通道
            blk.append(Residual(input_channels, num_channels, use_1x1conv=True, strides=2))
        else:
            # 后续块：保持形状
            blk.append(Residual(num_channels, num_channels))
    return nn.Sequential(*blk)

In [26]:
def get_resnet18(num_classes: int = 10) -> nn.Sequential:
    """构建完整的 ResNet-18 模型。"""
    
    # 阶段 1: Stem (初始卷积层)
    b1 = nn.Sequential(
        nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
        nn.BatchNorm2d(64), nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    )
    
    # 阶段 2-5: 残差阶段
    # 每个阶段通道翻倍，高宽减半
    b2 = resnet_block(64, 64, 2, first_block=True)
    b3 = resnet_block(64, 128, 2)
    b4 = resnet_block(128, 256, 2)
    b5 = resnet_block(256, 512, 2)
    
    # 阶段 6: 全局平均池化与输出
    net = nn.Sequential(
        b1, b2, b3, b4, b5,
        nn.AdaptiveAvgPool2d((1, 1)), # 特征提取器
        nn.Flatten(), # 展平到 (N, F) 形状
        nn.Linear(512, num_classes) # 分类器
    )
    return net


---

### 4. 细致梳理：ResNet 的核心知识点

1.  **相加 vs 拼接**：
    *   **ResNet** 使用的是 **Addition**（逐元素相加）。这要求两者 (`y`, `x`) 的形状完全一致。
    *   **GoogLeNet/DenseNet** 使用的是 **Concatenation**（通道拼接），通过 `torch.cat` 实现。
2.  **为什么能解决梯度消失？**
    *   在求导时，$\frac{\partial(g(x) + x)}{\partial x} = \frac{\partial g}{\partial x} + 1$。
    *   这个 **“+1”** 保证了即使 $g(x)$ 的梯度非常小，梯度依然能顺着这根电梯线传回底层，不会变成 0。
3.  **$1 \times 1$ 卷积的作用**：
    *   在 **Inception** 中主要是为了降维。
    *   在 **ResNet** 中主要是为了对齐形状。当我们需要改变通道数（如 64 变 128）或减小分辨率（stride=2）时，输入 $x$ 无法直接加到 $y$ 上。此时必须用 $1 \times 1$ 卷积对 $x$ 进行**线性投影**。
4.  **为什么没有全连接层 (不算最后的分类器)？**
    *   ResNet 继承了 NiN 的思想，使用 **Global Average Pooling** 替代了 AlexNet 中厚重的 FC 层。这大幅减少了参数量，降低了过拟合风险。

---

### 5. 关键工程暗知识 (Engineering Wisdom)

*   **Batch Norm 的位置**：ResNet 的标准做法是 `Conv -> BN -> ReLU`。注意，残差块的最后一次相加发生在第二个 BN 之后，ReLU 之前。
*   **初始化**：对于 ResNet，通常使用 **Kaiming 初始化**。
*   **性能**：ResNet-18 虽然比 VGG-11 深，但由于参数量分布均匀且去掉了巨大的全连接层，它的实际训练速度和显存占用表现非常优秀。

---

### 6. 总结 Checklist

*   **残差块**：理解 $f(x) = g(x) + x$ 的含义。
*   **跳跃连接**：明白其作为“梯度高速公路”的作用。
*   **1x1 投影**：掌握何时需要开启 `use_1x1conv`。
*   **层数计算**：ResNet-18 = 1(头) + 4阶段×2块×2层 + 1(尾) = 18 层。

---


## 7.7 稠密连接网络 (DenseNet)

> 如果说 ResNet 是通过“跨层加法”开辟了梯度的电梯直达通道，那么 **DenseNet**（由黄高教授等人提出）则是将这种思路推向了极致：
> 它不再是简单的跨层，而是让每一层都与后面**所有层**直接相连。

### 1. 核心动机：从“相加”到“连接”

#### 1.1 ResNet vs. DenseNet
*   **ResNet (Addition)**：将输入 $x$ 与残差 $f(x)$ 相加。在数学上，这就像是泰勒级数的展开，每一层只学习相对于前一层的微小增量。
*   **DenseNet (Concatenation)**：将输入 $x$ 与卷积输出 $f(x)$ 在**通道维度**上拼接。
    *   **公式**：$x_l = H_l([x_0, x_1, \dots, x_{l-1}])$
    *   **$H_l$**：是第 $l$ 层的非线性变换（通常包含 Batch Norm、ReLU 和 3x3 卷积）。
    *   **直觉**：每一层都能直接访问到之前所有层提取的原始特征，实现了极致的**特征复用**。

#### 1.2 优点
1.  **极强的梯度流**：误差信号可以直接传回任何一层，几乎没有梯度消失风险。
2.  **特征保留**：输入数据和早期的特征被明确地保留到了后面的层，网络不需要重新学习这些特征。
3.  **参数效率**：由于特征被反复复用，每一层只需要学习极少的新特征（通常只有 12-48 个通道），因此总参数量往往比 ResNet 更少。

    > 每一层产出的新特征图数量称为增长率 $k$。

---

### 2. 核心组件 

#### 2.1 稠密块 (Dense Block)

稠密块由多个卷积层组成，每一层的输出都会与输入拼接，作为下一层的输入。


In [ ]:
import torch
from torch import nn, Tensor

def conv_block(input_channels: int, num_channels: int) -> nn.Sequential:
    """构建 DenseNet 的基础卷积块。

    采用标准的 BN-ReLU-Conv 结构（即 Pre-activation 模式）。
    
    Args:
        input_channels: 输入通道数。
        num_channels: 输出的『增长率』。

    Returns:
        nn.Sequential: 包含规范化、激活和 3x3 卷积的序列。
    """
    return nn.Sequential(
        nn.BatchNorm2d(input_channels),
        nn.ReLU(),
        nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1)
    )

In [ ]:
class DenseBlock(nn.Module):
    """稠密块的实现。

    该模块内部的每一层都会接收之前所有层的输出拼接结果作为输入。
    """
    def __init__(self, num_convs: int, input_channels: int, growth_rate: int) -> None:
        """
        Args:
            num_convs: 块内包含的卷积层数量。
            input_channels: 初始输入通道数。
            growth_rate: 增长率（每一层输出的新通道数）。
        """
        super().__init__()
        self.net = nn.ModuleList() 
        for i in range(num_convs):
            # 每一层的输入通道 = 初始通道 + 之前所有层增加的通道
            in_c = input_channels + i * growth_rate
            self.net.append(conv_block(in_c, growth_rate))

    def forward(self, x: Tensor) -> Tensor:
        """前向传播：执行拼接逻辑。"""
        for blk in self.net:
            y = blk(x)
            # 在通道维度 (dim=1) 拼接：保留旧特征，加入新特征
            x = torch.cat((x, y), dim=1)
        return x

#### 2.2 过渡层 (Transition Layer)

由于 `DenseBlock` 会让通道数迅速膨胀（例如 10 层 32 增长率就会增加 320 个通道），且不改变宽高，我们需要一个“节制器”来压缩通道并减小分辨率。


In [ ]:
def transition_block(input_channels: int, num_channels: int) -> nn.Sequential:
    """过渡层：用于控制通道数增长并降低空间分辨率。

    Args:
        input_channels: 当前输入通道数。
        num_channels: 压缩后的输出通道数。

    Returns:
        nn.Sequential: 包含 1x1 卷积和平均池化的序列。
    """
    return nn.Sequential(
        nn.BatchNorm2d(input_channels),
        nn.ReLU(),
        # 1x1 卷积用于压缩通道数
        nn.Conv2d(input_channels, num_channels, kernel_size=1),
        # 2x2 平均池化用于减小宽高
        nn.AvgPool2d(kernel_size=2, stride=2)
    )


---

### 3. 构建 DenseNet 模型架构

In [ ]:
def get_densenet(num_classes: int = 10) -> nn.Sequential:
    """构建一个简单的 DenseNet 模型。"""
    
    # 1. Stem: 初始卷积层
    b1 = nn.Sequential(
        nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
        nn.BatchNorm2d(64), nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
    )
    
    # 2. Dense Blocks & Transition Layers
    # num_channels: 当前通道数
    # growth_rate: 每一层新增的通道数
    num_channels, growth_rate = 64, 32
    num_convs_in_dense_blocks = [4, 4, 4, 4] # 每个块里有 4 层卷积
    
    blks = []
    for i, num_convs in enumerate(num_convs_in_dense_blocks):
        blks.append(DenseBlock(num_convs, num_channels, growth_rate))
        # 更新进入下一个块之前的通道数
        num_channels += num_convs * growth_rate
        
        # 在 DenseBlock 之间添加过渡层，减半通道数
        if i != len(num_convs_in_dense_blocks) - 1:
            blks.append(transition_block(num_channels, num_channels // 2))
            num_channels //= 2
            
    # 3. Head: 全局池化与全连接
    return nn.Sequential(
        b1, *blks,
        nn.BatchNorm2d(num_channels), nn.ReLU(),
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(num_channels, num_classes)
    )


---

### 4. 细致梳理：DenseNet 的关键知识点

1.  **增长率 (Growth Rate)**：
    *   这是 DenseNet 特有的超参数 $k$。它控制了每一层“贡献”了多少新知识。
    *   即便 $k$ 很小（如 12），因为有之前所有层的积累，网络依然非常强大。
2.  **瓶颈层 (Bottleneck)**：
    *   虽然在上面的代码中为了简洁没写，但标准 DenseNet（DenseNet-B）在 $3 \times 3$ 卷积前会加一个 $1 \times 1$ 卷积，将输入通道减少到 $4k$，以提高计算效率。
3.  **压缩 (Compression)**：
    *   在过渡层中，我们将通道数减少（如减少一半），这被称为 DenseNet-C。同时具备瓶颈层和压缩的版本称为 **DenseNet-BC**。

---

### 5. 关键工程暗知识 (Engineering Wisdom)

*   **内存陷阱**：DenseNet 虽然参数少，但由于需要存储大量的中间拼接张量，**训练时的显存占用通常比 ResNet 高**。在显存有限的情况下，ResNet 往往是更安全的选择。
*   **特征复用可视化**：研究表明，DenseNet 的深层确实会直接使用非常浅层的原始像素特征。

---

### 6. 补充知识：DenseNet 的历史地位与工程缺陷

*   **架构地位：ResNet 连接思想的“逻辑终点”**
    *   **极致复用**：如果说 ResNet 引入了“跨层连接”，那么 DenseNet 则将其推向了极致（**稠密连接**）。它在学术上证明了：通过**通道拼接 (Concatenation)** 显式保留原始特征，比相加 (Addition) 融合更能榨干每一层参数的价值。
    *   **窄模型先驱**：它打破了“层数多必宽”的偏见，通过微小的增长率 $k$ 实现了“极窄且深”的高效架构。

*   **致命缺陷：显存与计算的“隐形杀手”**
    *   **显存黑洞**：由于反向传播需要存储每一层拼接后的巨量中间张量，**训练显存占用极高**（远超同规模 ResNet）。在显存受限时，ResNet 仍是工业界的首选。
    *   **IO 瓶颈**：频繁的 `torch.cat` 操作涉及大量的内存拷贝（Memory Copy），这会导致在某些硬件（如移动端 DSP/NPU）上的**推理延迟 (Latency) 显著增加**。
    *   **并非“越深越好”**：随着深度增加，拼接后的通道数会迅速膨胀，导致参数量和计算量呈二次方增长，限制了它像 ResNet 那样轻松扩展到 1000 层。

*   **小结：** DenseNet 是**小数据集上的泛化神器**（抗过拟合强），但在**大规模工业落地中受限于内存效率**。

---

### 7. 总结 Checklist

*   **拼接 (Cat)**：理解它与 ResNet 相加 (Add) 的本质区别。
*   **稠密连接**：明白为什么第 $L$ 层有 $L$ 个输入连接。
*   **过渡层**：掌握其下采样和通道压缩的双重作用。
*   **参数量**：理解为什么特征复用能让网络变得“窄而深”。

---


## 附录：规范化家族 (Normalization Family)

> 虽然 **Batch Normalization (BN)** 在卷积神经网络（CNN）中统治了多年，但它有两个致命的弱点：
> 1.  **强依赖 Batch Size**：如果 Batch Size 太小（比如显存不够，只能设为 2），统计出的均值和方差会极度不稳定，导致模型崩溃。
> 2.  **不适用于变长序列**：在循环神经网络（RNN）或 Transformer 中，不同样本长度不同，Batch 维度的统计变得非常困难。
>
> 为了解决这些问题，深度学习领域演化出了一个 **“规范化家族”**。
> 
> 理解它们的关键在于：**看它们是在张量的哪几个维度上“切那一刀”算均值和方差。**

---

### 1. 核心维度的定义

假设我们的输入张量形状为 $(N, C, H, W)$：
*   **$N$**: Batch（批大小）
*   **$C$**: Channels（通道数）
*   **$H, W$**: Spatial（空间高度和宽度）

---

### 2. 四大主流规范化操作

#### 2.1 Layer Normalization (LN) —— Transformer 的基石
*   **切法**：对**单个样本**的所有特征进行规范化。
*   **数学逻辑**：在 $(C, H, W)$ 维度上算均值方差。
*   **优点**：完全独立于 Batch Size；在 NLP（如 GPT, BERT）中表现极佳。

#### 2.2 Instance Normalization (IN) —— 风格迁移神器
*   **切法**：对**每个样本的每个通道**独立规范化。
*   **数学逻辑**：在 $(H, W)$ 维度上算均值方差。
*   **应用**：图像风格迁移（Style Transfer），因为它可以消除图像整体亮度和对比度的影响。

#### 2.3 Group Normalization (GN) —— 小 Batch 的救星
*   **切法**：介于 LN 和 IN 之间。将通道 $C$ 分成若干组（Groups），在每组内算规范化。
*   **应用**：目标检测（Object Detection）等显存消耗巨大的任务（此时 Batch Size 通常只能设为 1 或 2）。

#### 2.4 RMSNorm —— 大模型（Llama）标配
*   **简化版 LN**：不减均值，只除以**均方根（Root Mean Square）**。
*   **优点**：计算更简单，性能几乎无损，目前主流大语言模型（LLM）都在用。

---

### 3. 规范化层工厂

在工程中，我们通常需要根据配置灵活切换不同的规范化层。


In [24]:
import torch
from torch import nn, Tensor
from typing import Union, Optional

class NormFactory:
    """模块化规范化层工厂类。"""

    @staticmethod
    def get_norm_layer(
        norm_type: str, 
        num_channels: int, 
        num_groups: Optional[int] = 32
    ) -> nn.Module:
        """根据类型返回对应的规范化层。

        Args:
            norm_type: 规范化类型 ('bn', 'ln', 'in', 'gn')。
            num_channels: 输入的通道数。
            num_groups: 仅用于 GN，表示分组数。

        Returns:
            nn.Module: 初始化的规范化层。
        """
        norm_type = norm_type.lower()
        
        if norm_type == 'bn':
            # 针对 4D 图像输入
            return nn.BatchNorm2d(num_channels)
        elif norm_type == 'ln':
            # LayerNorm 需要指定规范化的形状，这里假设对后三维规范化
            # 注意：实际工程中 LN 形状需根据特征图大小调整
            return nn.GroupNorm(1, num_channels) # GN 分组为 1 等价于 LN
        elif norm_type == 'in':
            return nn.InstanceNorm2d(num_channels)
        elif norm_type == 'gn':
            return nn.GroupNorm(num_groups, num_channels)
        else:
            raise ValueError(f"Unsupported norm type: {norm_type}")

In [25]:
# --- 验证各种 Norm 的输出形状 ---
X = torch.randn(2, 64, 32, 32) # (N, C, H, W)

for n_type in ['bn', 'in', 'gn']:
    layer = NormFactory.get_norm_layer(n_type, 64)
    output = layer(X)
    print(f"{n_type.upper():<4} 输出形状: {output.shape}")

BN   输出形状: torch.Size([2, 64, 32, 32])
IN   输出形状: torch.Size([2, 64, 32, 32])
GN   输出形状: torch.Size([2, 64, 32, 32])



---

### 4. 深度对比：那一刀是怎么切的？

| 类型 | 均值方差的计算范围 | 独立于 Batch？ | 典型场景 |
| :--- | :--- | :--- | :--- |
| **BN** | $(N, H, W)$ | **否** | 经典的 CNN (ResNet, VGG) |
| **LN** | $(C, H, W)$ | **是** | Transformer, RNN |
| **IN** | $(H, W)$ | **是** | GAN, 风格迁移 |
| **GN** | $(C/G, H, W)$ | **是** | 显存受限的检测/分割任务 |

---

### 5. 关键工程暗知识 (Engineering Wisdom)

1.  **LayerNorm 的特殊性**：
    在 PyTorch 中，`nn.LayerNorm` 通常用于 NLP 的 3D 张量 $(N, L, C)$。对于 CV 的 4D 张量，直接用 `nn.LayerNorm` 会很麻烦，因为它的实现是针对最后几个维度的。工程上常用 `nn.GroupNorm(1, num_channels)` 来完美替代 4D 的 LayerNorm。
2.  **同步批量规范化 (SyncBN)**：
    在多卡并行训练（DDP）时，单卡的 Batch Size 可能只有 2。此时标准的 BN 效果很差。工程上会使用 `nn.SyncBatchNorm`，让多张显卡之间通信，共同计算一个“大 Batch”的均值和方差。
3.  **推理时的坑**：
    除了 BN 以外，LN、IN 和 GN 在**训练和测试时的行为是完全一致的**（都直接算当前输入的统计量）。只有 BN 需要维护 `moving_mean`。

---

### 6. 总结 Checklist

*   **BN**：跨样本算，怕小 Batch。
*   **LN**：样本内算，NLP 标配。
*   **IN**：通道内算，去风格化。
*   **GN**：分组算，BN 的小 Batch 替代品。

---
